# Import

In [1]:
import os
import sys
import copy
import time
import math
import hashlib
import json
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import numpy as np
base = Path('/ix/rbao/Projects/HCC-CBS-175-Hillman-RBao-panCancer-HE/')
data = base.joinpath('results','infer_outputs','tile_dfs')
print(data.exists())

True


In [4]:
df = pd.read_csv(fns[0],
                     sep='\t')

In [5]:
df.head()

,tile,x,y,sz,cur_path,p_pos,pred_cls
0,S08-4387_n5677_x73696_y3136_px224.jpg,73696.0,3136.0,224.0,/scratch/slurm-2203027/results/tiles/S08-4387/...,0.001319,1
1,S08-4387_n5678_x73920_y3136_px224.jpg,73920.0,3136.0,224.0,/scratch/slurm-2203027/results/tiles/S08-4387/...,0.000214,1
2,S08-4387_n5679_x74144_y3136_px224.jpg,74144.0,3136.0,224.0,/scratch/slurm-2203027/results/tiles/S08-4387/...,0.003203,1
3,S08-4387_n5680_x74368_y3136_px224.jpg,74368.0,3136.0,224.0,/scratch/slurm-2203027/results/tiles/S08-4387/...,0.011879,1
4,S08-4387_n5681_x74592_y3136_px224.jpg,74592.0,3136.0,224.0,/scratch/slurm-2203027/results/tiles/S08-4387/...,0.000828,1


In [23]:
print(np.sum(df.p_pos > 0.5), np.sum(df.pred_cls==0))

5165 5165


In [19]:
0.4964/2

0.2482

In [20]:
def tiles_to_area(df):
    n = df.shape[0]
    sz = df.sz[0] * 0.2517 #20x in microns
    area = (n * sz**2) / 1e6 #mm2
    return area

In [25]:
fns = [x for x in data.glob('*.tsv')]
print(len(fns))
p_df = pd.DataFrame([])
for i,fn in enumerate(tqdm(fns)):
    df = pd.read_csv(fn,
                     sep='\t')
    tot_n = df.shape[0]
    pos = np.sum(df.pred_cls == 0)
    per = (pos/tot_n) * 100
    slide = Path(df.cur_path[0]).parts[-1].split('_n')[0] + '.svs'
    p_df.loc[i,'slide'] = slide
    p_df.loc[i,'percent_predicted_tumor'] = per
    p_df.loc[i,'total_area_mm2'] = tiles_to_area(df)
    sub = df.loc[df.pred_cls == 0,:].reset_index(drop=True)
    p_df.loc[i,'pred_tum_area_mm2'] = tiles_to_area(sub)
    # idx = df.pred_cls==0
    

44


100%|██████████| 44/44 [00:02<00:00, 18.72it/s]


In [31]:
p_df.sort_values('pred_tum_area_mm2',ascending=False).reset_index(drop=True)

,slide,percent_predicted_tumor,total_area_mm2,pred_tum_area_mm2
0,S13-18183.svs,76.542246,162.830575,124.634179
1,S15-5437.svs,42.882177,170.510543,73.118634
2,S16-8435.svs,35.342778,195.209777,68.992558
3,S11-732.svs,34.295936,184.818297,63.385164
4,S08-4387.svs,39.673830,124.551530,49.414362
5,S12-13048.svs,23.224852,154.718291,35.933094
6,S17-19517.svs,21.618611,151.259763,32.700260
7,S08-978.svs,17.450315,163.946332,28.609151
8,S06-4083.svs,11.865095,219.044379,25.989825
9,S10-15984.svs,41.998699,39.092816,16.418474


In [27]:
p_df = p_df.sort_values('percent_predicted_tumor',
                ascending=False).reset_index(drop=True)
p_df.head()
results = base.joinpath('results','infer_outputs')
fn = results.joinpath(
                    'acral_percent_tumor_predictions_%d_slides.csv' % p_df.shape[0]
                    )
print(fn)
p_df.to_csv(fn)


/ix/rbao/Projects/HCC-CBS-175-Hillman-RBao-panCancer-HE/results/infer_outputs/acral_percent_tumor_predictions_44_slides.csv
